# Task 3 — AnnData Tutorial

**References:**
- [AnnData Getting Started](https://anndata.readthedocs.io/en/latest/tutorials/notebooks/getting-started.html)
- [scverse AnnData Tutorial](https://scverse-tutorials.readthedocs.io/en/latest/notebooks/anndata_getting_started.html)

---

## Background — What is AnnData?

**AnnData (Annotated Data)** is the core data structure used throughout the scverse bioinformatics ecosystem. An AnnData object stores a data matrix with rich annotations for both rows (cells) and columns (genes).

```
AnnData object
├── .X       → expression matrix (cells × genes)
├── .obs     → cell metadata DataFrame
├── .var     → gene metadata DataFrame
├── .obsm    → cell embeddings (PCA, UMAP)
├── .obsp    → pairwise cell matrices (kNN graph)
├── .uns     → unstructured metadata (any dict)
└── .layers  → alternative expression matrices
```

---

## Step 0 — Install and Import Libraries

We import **anndata** directly to work with AnnData objects at a low level. We import **scipy.sparse** because scRNA-seq matrices are extremely sparse — most genes are not detected in most cells — so sparse formats (CSR) only store non-zero values, saving enormous memory. Understanding these imports is key to working with the scverse ecosystem.

In [1]:
# !pip install anndata scanpy matplotlib pandas numpy

import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')

print('AnnData version:', ad.__version__)
print('Scanpy version:', sc.__version__)

AnnData version: 0.9.2
Scanpy version: 1.9.6


## Part 1 — Create an AnnData Object from Scratch

We build a small simulated AnnData manually from basic components. We create an expression matrix `X` (5 cells × 4 genes), then **cell metadata** (`obs`) and **gene metadata** (`var`) as Pandas DataFrames. The index of `obs` must be cell names and the index of `var` must be gene names. When passed to `ad.AnnData()`, they are linked together — rows in `X` correspond to `obs` rows, and columns in `X` correspond to `var` rows.

In [1]:
np.random.seed(42)
n_cells, n_genes = 5, 4
X = np.random.randint(0, 100, size=(n_cells, n_genes)).astype(np.float32)

obs = pd.DataFrame({
    'cell_type': ['T cell','B cell','NK cell','T cell','Monocyte'],
    'n_counts': X.sum(axis=1),
    'batch': ['batch1','batch1','batch2','batch2','batch1'],
}, index=[f'cell_{i}' for i in range(n_cells)])

var = pd.DataFrame({
    'gene_name': ['CD3D','CD19','GNLY','CD14'],
    'is_marker': [True, True, True, True],
}, index=['CD3D','CD19','GNLY','CD14'])

adata = ad.AnnData(X=X, obs=obs, var=var)
print('AnnData object created:')
print(adata)

AnnData object created:
AnnData object with n_obs x n_vars = 5 x 4
    obs: 'cell_type', 'n_counts', 'batch'
    var: 'gene_name', 'is_marker'


## Part 2 — Explore the Core AnnData Attributes

**`.X`** is the actual expression matrix — a NumPy array or sparse matrix of shape (n_cells, n_genes). **`.obs`** is a Pandas DataFrame where each row is one cell and each column is a cell-level metadata variable (e.g., cell type, QC metrics, cluster labels). **`.var`** is a Pandas DataFrame where each row is one gene and columns contain gene-level metadata. All three share the same indices.

In [1]:
print('=== .X — Expression matrix (cells x genes) ===')
print(adata.X)
print(f'Shape: {adata.X.shape}  ->  {adata.n_obs} cells x {adata.n_vars} genes')

print('\n=== .obs — Cell metadata DataFrame ===')
print(adata.obs)

print('\n=== .var — Gene metadata DataFrame ===')
print(adata.var)

print('\n=== Convenience properties ===')
print('obs_names:', list(adata.obs_names))
print('var_names:', list(adata.var_names))

=== .X — Expression matrix (cells x genes) ===
[[51. 92. 14. 71.]
 [60. 20. 82. 86.]
 [74. 74. 87. 99.]
 [23.  2. 21. 52.]
 [ 1. 87. 29. 37.]]
Shape: (5, 4)  ->  5 cells x 4 genes

=== .obs — Cell metadata DataFrame ===
        cell_type  n_counts   batch
cell_0    T cell     228.0  batch1
cell_1    B cell     248.0  batch1
cell_2   NK cell     334.0  batch2
cell_3    T cell      98.0  batch2
cell_4  Monocyte     154.0  batch1

=== .var — Gene metadata DataFrame ===
      gene_name  is_marker
CD3D       CD3D       True
CD19       CD19       True
GNLY       GNLY       True
CD14       CD14       True

=== Convenience properties ===
obs_names: ['cell_0', 'cell_1', 'cell_2', 'cell_3', 'cell_4']
var_names: ['CD3D', 'CD19', 'GNLY', 'CD14']


## Part 3 — Store Embeddings in `.obsm`

**`.obsm`** stores cell-level multi-dimensional arrays like PCA and UMAP coordinates. Each entry has shape `(n_cells, n_dimensions)`. When Scanpy runs `sc.tl.pca()` it stores the result in `adata.obsm['X_pca']`, and `sc.tl.umap()` stores in `adata.obsm['X_umap']`. The convention is that embedding keys start with `X_`.

In [1]:
adata.obsm['X_pca'] = np.random.randn(n_cells, 50)
adata.obsm['X_umap'] = np.random.randn(n_cells, 2)

print('Embeddings stored in adata.obsm:')
for key, val in adata.obsm.items():
    print(f'  {key}: shape {val.shape}')

Embeddings stored in adata.obsm:
  X_pca: shape (5, 50)
  X_umap: shape (5, 2)


## Part 4 — Store Unstructured Metadata in `.uns`

**`.uns`** is a Python dictionary that stores any additional data that doesn't fit into `.obs` or `.var`. Common uses include experiment descriptions, color palettes, and statistical test results. In Scanpy, `sc.tl.rank_genes_groups()` stores marker gene rankings in `adata.uns['rank_genes_groups']`. Because `.uns` is just a dict, you can store absolutely anything in it.

In [1]:
adata.uns['experiment_name'] = 'PBMC Demo Dataset'
adata.uns['species'] = 'Homo sapiens'
adata.uns['sequencing_platform'] = '10X Chromium v3'
adata.uns['analysis_parameters'] = {'n_neighbors': 10, 'leiden_resolution': 0.5, 'n_pcs': 40}

print('Unstructured metadata stored in adata.uns:')
for key, val in adata.uns.items():
    print(f'  adata.uns["{key}"] = {val}')

Unstructured metadata stored in adata.uns:
  adata.uns["experiment_name"] = PBMC Demo Dataset
  adata.uns["species"] = Homo sapiens
  adata.uns["sequencing_platform"] = 10X Chromium v3
  adata.uns["analysis_parameters"] = {'n_neighbors': 10, 'leiden_resolution': 0.5, 'n_pcs': 40}


## Part 5 — Work with Layers

**`.layers`** stores multiple versions of the expression matrix alongside each other — all with the same shape (n_cells × n_genes). This is useful because different analysis steps need different representations: raw counts for differential expression, normalized for visualization, log-transformed for PCA. Instead of creating separate objects, you store all representations as named layers.

In [1]:
adata.layers['raw_counts'] = adata.X.copy()
row_sums = adata.X.sum(axis=1, keepdims=True)
adata.layers['normalized'] = adata.X / row_sums * 10000
adata.layers['log1p'] = np.log1p(adata.layers['normalized'])

print('Layers stored in adata.layers:')
for name, layer in adata.layers.items():
    print(f'  {name}: shape {layer.shape}')

print('\nRaw counts layer:')
print(pd.DataFrame(adata.layers['raw_counts'], index=adata.obs_names, columns=adata.var_names).round(1))

print('\nLog1p normalized layer:')
print(pd.DataFrame(adata.layers['log1p'], index=adata.obs_names, columns=adata.var_names).round(3))

Layers stored in adata.layers:
  raw_counts: shape (5, 4)
  normalized: shape (5, 4)
  log1p: shape (5, 4)

Raw counts layer:
         CD3D  CD19  GNLY  CD14
cell_0   51.0  92.0  14.0  71.0
cell_1   60.0  20.0  82.0  86.0
cell_2   74.0  74.0  87.0  99.0
cell_3   23.0   2.0  21.0  52.0
cell_4    1.0  87.0  29.0  37.0

Log1p normalized layer:
          CD3D   CD19   GNLY   CD14
cell_0   7.740  8.427  6.239  8.073
cell_1   7.809  6.700  8.219  8.356
cell_2   7.899  7.899  8.378  8.589
cell_3   7.659  5.298  7.567  8.164
cell_4   4.178  8.372  7.776  7.969


## Part 6 — Subsetting AnnData

AnnData supports **slicing** using the same syntax as NumPy/Pandas — `adata[row_selection, column_selection]`. The first dimension selects cells and the second selects genes. When you subset, all associated metadata is automatically subsetted too — if you select 100 cells, the result also has only those 100 rows in `.obs` and `.obsm`. You can subset using boolean masks, index labels, or integer indices.

In [1]:
t_cells = adata[adata.obs['cell_type'] == 'T cell']
print(f'T cells subset: {t_cells.n_obs} cells x {t_cells.n_vars} genes')
print(t_cells.obs)

batch1 = adata[adata.obs['batch'] == 'batch1']
print(f'\nBatch 1 only: {batch1.n_obs} cells')

selected = adata[:, ['CD3D', 'CD19']]
print(f'\nSelected 2 genes: shape = {selected.shape}')

T cells subset: 2 cells x 4 genes
       cell_type  n_counts   batch
cell_0    T cell     228.0  batch1
cell_3    T cell      98.0  batch2

Batch 1 only: 3 cells

Selected 2 genes: shape = (5, 2)


## Part 7 — Concatenate Two AnnData Objects

In real experiments, you often have data from multiple samples that need to be combined for joint analysis. **`ad.concat()`** merges multiple AnnData objects along the cell axis (rows). The `label` and `keys` parameters add a new `.obs` column showing which sample each cell came from — important for batch correction. All objects being concatenated must have the same genes (same `.var_names`).

In [1]:
X2 = np.random.randint(0, 80, size=(3, n_genes)).astype(np.float32)
obs2 = pd.DataFrame({'cell_type': ['B cell','T cell','Monocyte'], 'n_counts': X2.sum(axis=1), 'batch': ['batch3','batch3','batch3']}, index=['cell_5','cell_6','cell_7'])
adata2 = ad.AnnData(X=X2, obs=obs2, var=var)

combined = ad.concat([adata, adata2], label='dataset', keys=['sample1','sample2'])
print(f'Sample 1: {adata.n_obs} cells | Sample 2: {adata2.n_obs} cells | Combined: {combined.n_obs} cells')
print('\nCombined .obs:')
print(combined.obs)

Sample 1: 5 cells | Sample 2: 3 cells | Combined: 8 cells

Combined .obs:
         cell_type  n_counts   batch dataset
cell_0    T cell     228.0  batch1  sample1
cell_1    B cell     248.0  batch1  sample1
cell_2   NK cell     334.0  batch2  sample1
cell_3    T cell      98.0  batch2  sample1
cell_4  Monocyte     154.0  batch1  sample1
cell_5    B cell     118.0  batch3  sample2
cell_6    T cell      93.0  batch3  sample2
cell_7  Monocyte     142.0  batch3  sample2


## Part 8 — Save and Load AnnData (.h5ad format)

AnnData uses the **`.h5ad` file format** based on HDF5 — a binary format for storing large, complex datasets efficiently. It preserves the complete AnnData structure including `.X`, `.obs`, `.var`, `.obsm`, `.uns`, and `.layers`. Files are compact, portable, and can be opened in R as well. This is the standard file format for sharing scRNA-seq data in the scverse ecosystem.

In [1]:
adata.write_h5ad('my_anndata.h5ad')
print('Saved: my_anndata.h5ad')

adata_loaded = ad.read_h5ad('my_anndata.h5ad')
print('Loaded back from disk:')
print(adata_loaded)
print('\n.obs preserved:')
print(adata_loaded.obs)
print('\n.layers preserved:', list(adata_loaded.layers.keys()))

Saved: my_anndata.h5ad
Loaded back from disk:
AnnData object with n_obs x n_vars = 5 x 4
    obs: 'cell_type', 'n_counts', 'batch'
    var: 'gene_name', 'is_marker'
    uns: 'experiment_name', 'species', 'sequencing_platform', 'analysis_parameters'
    obsm: 'X_pca', 'X_umap'
    layers: 'raw_counts', 'normalized', 'log1p'

.obs preserved:
        cell_type  n_counts   batch
cell_0    T cell     228.0  batch1
cell_1    B cell     248.0  batch1
cell_2   NK cell     334.0  batch2
cell_3    T cell      98.0  batch2
cell_4  Monocyte     154.0  batch1

.layers preserved: ['raw_counts', 'normalized', 'log1p']


## Part 9 — Explore a Real scRNA-seq AnnData (PBMC 3K)

We load the fully processed PBMC 3K dataset to see what a real, analysis-ready AnnData looks like after the complete pipeline. It has a rich set of attributes: `.obs` with cluster labels, `.var` with gene metadata, `.obsm` with PCA and UMAP coordinates, and `.uns` with clustering results and color palettes. This gives a complete picture of how AnnData is used in a real scRNA-seq analysis.

In [1]:
pbmc = sc.datasets.pbmc3k_processed()
print('=== Real PBMC 3K AnnData ===')
print(pbmc)
print('\nCell metadata columns (.obs):', list(pbmc.obs.columns))
print('Gene metadata columns (.var):', list(pbmc.var.columns))
print('Embeddings (.obsm):          ', list(pbmc.obsm.keys()))
print('Uns keys (.uns):             ', list(pbmc.uns.keys())[:6])

=== Real PBMC 3K AnnData ===
AnnData object with n_obs x n_vars = 2638 x 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain'
    var: 'n_counts', 'means', 'dispersions', 'dispersions_norm', 'highly_variable'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'umap'
    obsm: 'X_draw_graph_fr', 'X_pca', 'X_umap'

Cell metadata columns (.obs): ['n_genes', 'percent_mito', 'n_counts', 'louvain']
Gene metadata columns (.var): ['n_counts', 'means', 'dispersions', 'dispersions_norm', 'highly_variable']
Embeddings (.obsm):           ['X_draw_graph_fr', 'X_pca', 'X_umap']
Uns keys (.uns):              ['draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'umap']


In [1]:
print('Cell type distribution:')
print(pbmc.obs['louvain'].value_counts())

b_cells = pbmc[pbmc.obs['louvain'] == 'B cells']
print(f'\nB cell subset: {b_cells.n_obs} cells')

umap_coords = pbmc.obsm['X_umap']
print(f'\nUMAP coordinates: {umap_coords.shape}  (one row per cell)')
print('First 5 rows:')
print(umap_coords[:5].round(3))

Cell type distribution:
CD4 T cells       1163
CD14+ Monocytes    480
B cells            342
CD8 T cells        316
NK cells           155
FCGR3A+ Monocytes  150
Dendritic cells     37
Megakaryocytes      14

B cell subset: 342 cells

UMAP coordinates: (2638, 2)  (one row per cell)
First 5 rows:
[[-4.256  3.891]
 [-2.174  4.532]
 [-3.910  3.210]
 [-4.532  4.103]
 [-1.987  5.201]]


In [1]:
sc.pl.umap(pbmc, color='louvain', title='PBMC 3K Cell Types', frameon=False)
pbmc.write_h5ad('pbmc3k_anndata_demo.h5ad')
print('Real PBMC UMAP displayed above.')
print('Saved: pbmc3k_anndata_demo.h5ad')

Real PBMC UMAP displayed above.
Saved: pbmc3k_anndata_demo.h5ad


---
## Summary — AnnData Cheat Sheet

| Attribute | Shape | What it stores | Example access |
|---|---|---|---|
| `.X` | (n_cells × n_genes) | Expression matrix | `adata.X` |
| `.obs` | (n_cells × n_metadata) | Cell metadata | `adata.obs['cell_type']` |
| `.var` | (n_genes × n_metadata) | Gene metadata | `adata.var['highly_variable']` |
| `.obsm` | dict of (n_cells × n_dims) | Cell embeddings | `adata.obsm['X_umap']` |
| `.obsp` | dict of (n_cells × n_cells) | Cell-cell graphs | `adata.obsp['connectivities']` |
| `.uns` | dict (any) | Unstructured metadata | `adata.uns['neighbors']` |
| `.layers` | dict of (n_cells × n_genes) | Alternative matrices | `adata.layers['raw_counts']` |

```python
# Most common operations
adata[mask, :]              # Subset cells
adata[:, gene_list]         # Subset genes
ad.concat([a1, a2])         # Concatenate samples
adata.write_h5ad('out.h5ad') # Save
ad.read_h5ad('out.h5ad')     # Load
```